In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/processed/recalls.duckdb")
def q(sql):
    return con.execute(sql).df()

CMPL_COLS = [
    "CMPLID", "ODINO", "MFR_NAME", "MAKETXT", "MODELTXT", "YEARTXT", "CRASH", "FAILDATE",
    "FIRE", "INJURED", "DEATHS", "COMPDESC", "CITY", "STATE", "VIN", "DATEA", "LDATE",
    "MILES", "OCCURENCES", "CDESCR", "CMPL_TYPE", "POLICE_RPT_YN", "PURCH_DT",
    "ORIG_OWNER_YN", "ANTI_BRAKES_YN", "CRUISE_CONT_YN", "NUM_CYLS", "DRIVE_TRAIN",
    "FUEL_SYS", "FUEL_TYPE", "TRANS_TYPE", "VEH_SPEED", "DOT", "TIRE_SIZE", "LOC_OF_TIRE",
    "TIRE_FAIL_TYPE", "ORIG_EQUIP_YN", "MANUF_DT", "SEAT_TYPE", "RESTRAINT_TYPE",
    "DEALER_NAME", "DEALER_TEL", "DEALER_CITY", "DEALER_STATE", "DEALER_ZIP", "PROD_TYPE",
    "REPAIRED_YN", "MEDICAL_ATTN", "VEHICLES_TOWED_YN", "STATE_OF_INCIDENT",
    "VEHICLE_OPERATOR",
]
RCL_COLS = [
    "RECORD_ID", "CAMPNO", "MAKETXT", "MODELTXT", "YEARTXT", "MFGCAMPNO", "COMPNAME",
    "MFGNAME", "BGMAN", "ENDMAN", "RCLTYPECD", "POTAFF", "ODATE", "INFLUENCED_BY",
    "MFGTXT", "RCDATE", "DATEA", "RPNO", "FMVSS", "DESC_DEFECT", "CONEQUENCE_DEFECT",
    "CORRECTIVE_ACTION", "NOTES", "RCL_CMPT_ID", "MFR_COMP_NAME", "MFR_COMP_DESC",
    "MFR_COMP_PTNO", "DO_NOT_DRIVE", "PARK_OUTSIDE",
]
print(len(CMPL_COLS), len(RCL_COLS))

51 29


In [2]:
READ_OPTS = "delim='\t', header=false, quote='', escape='', all_varchar=true, null_padding=true, strict_mode=false"
names = "[" + ", ".join(f"'{c}'" for c in CMPL_COLS) + "]"

con.execute(f"""
CREATE OR REPLACE TABLE complaints_raw AS
SELECT * FROM read_csv('../data/raw/COMPLAINTS_RECEIVED_*.txt', {READ_OPTS}, names={names})
""")

print(q("SELECT COUNT(*) AS rows FROM complaints_raw").to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows
1119391


In [3]:
# A valid LDATE is exactly 8 digits. Anything else means a broken or shifted row.
print("Rows with an invalid received date:")
print(q("""
SELECT COUNT(*) AS bad_rows,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM complaints_raw), 3) AS pct
FROM complaints_raw
WHERE NOT regexp_matches(COALESCE(LDATE, ''), '^\\d{8}$')
""").to_string(index=False))

print("\nProduct types (V = vehicle):")
print(q("SELECT PROD_TYPE, COUNT(*) AS n FROM complaints_raw GROUP BY 1 ORDER BY 2 DESC").to_string(index=False))

print("\nComplaint sources:")
print(q("SELECT CMPL_TYPE, COUNT(*) AS n FROM complaints_raw GROUP BY 1 ORDER BY 2 DESC").to_string(index=False))

Rows with an invalid received date:
 bad_rows  pct
        0  0.0

Product types (V = vehicle):
PROD_TYPE       n
        V 1106034
        T    7024
        E    3679
        C    2639
      NaN      15

Complaint sources:
CMPL_TYPE      n
     IVOQ 869480
     EVOQ 239871
     LETR   7628
      CON    764
     SVOQ    690
      VOQ    451
     MIVQ    382
     MAVQ    117
       DP      8


In [4]:
con.execute("""
CREATE OR REPLACE TABLE complaints AS
SELECT
    CMPLID, ODINO,
    UPPER(TRIM(MAKETXT))                  AS make,
    UPPER(TRIM(MODELTXT))                 AS model,
    TRY_CAST(YEARTXT AS INTEGER)          AS model_year,
    UPPER(TRIM(COMPDESC))                 AS component,
    try_strptime(LDATE, '%Y%m%d')::DATE   AS received_date,
    try_strptime(FAILDATE, '%Y%m%d')::DATE AS fail_date,
    CRASH = 'Y'                           AS crash,
    FIRE = 'Y'                            AS fire,
    TRY_CAST(INJURED AS INTEGER)          AS injured,
    TRY_CAST(DEATHS AS INTEGER)           AS deaths,
    MEDICAL_ATTN = 'Y'                    AS medical_attn,
    VEHICLES_TOWED_YN = 'Y'               AS towed,
    TRY_CAST(MILES AS INTEGER)            AS miles,
    CMPL_TYPE                             AS source,
    CDESCR                                AS narrative
FROM complaints_raw
WHERE PROD_TYPE = 'V'                                   -- vehicles only
  AND regexp_matches(COALESCE(LDATE, ''), '^\\d{8}$')   -- drop broken rows
  AND COALESCE(CMPL_TYPE, '') <> 'RC'                   -- recall-generated: leakage
  AND TRY_CAST(YEARTXT AS INTEGER) BETWEEN 1990 AND 2027
""")

print(q("""
SELECT EXTRACT(year FROM received_date) AS yr, COUNT(*) AS complaints
FROM complaints GROUP BY 1 ORDER BY 1
""").to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  yr  complaints
2015      103838
2016       94436
2017       90330
2018       89847
2019      104976
2020       82806
2021       68084
2022       75190
2023       90306
2024       96397
2025      108126
2026       99308


In [5]:
rnames = "[" + ", ".join(f"'{c}'" for c in RCL_COLS) + "]"

con.execute(f"""
CREATE OR REPLACE TABLE recalls AS
SELECT
    CAMPNO                                AS campaign,
    UPPER(TRIM(MAKETXT))                  AS make,
    UPPER(TRIM(MODELTXT))                 AS model,
    TRY_CAST(YEARTXT AS INTEGER)          AS model_year,
    UPPER(TRIM(COMPNAME))                 AS component,
    try_strptime(RCDATE, '%Y%m%d')::DATE  AS recall_date,
    TRY_CAST(POTAFF AS BIGINT)            AS units_affected,
    INFLUENCED_BY                         AS initiated_by,
    DESC_DEFECT                           AS defect_summary
FROM read_csv('../data/raw/FLAT_RCL_POST_2010.txt', {READ_OPTS}, names={rnames})
WHERE RCLTYPECD = 'V'
""")

print(q("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT campaign) AS campaigns,
       MIN(recall_date) AS first, MAX(recall_date) AS last
FROM recalls
""").to_string(index=False))

print(q("""
SELECT initiated_by, COUNT(DISTINCT campaign) AS campaigns
FROM recalls GROUP BY 1 ORDER BY 2 DESC
""").to_string(index=False))

  rows  campaigns      first       last
220369      13738 2010-01-01 2026-10-29
initiated_by  campaigns
         MFR      12276
         ODI       1353
        OVSC        109


In [6]:
print("Top complaint components:")
print(q("SELECT component, COUNT(*) AS n FROM complaints GROUP BY 1 ORDER BY 2 DESC LIMIT 15").to_string(index=False))

print("\nTop recall components:")
print(q("SELECT component, COUNT(DISTINCT campaign) AS n FROM recalls GROUP BY 1 ORDER BY 2 DESC LIMIT 15").to_string(index=False))

Top complaint components:
                                               component      n
                                                  ENGINE 148471
                                       ELECTRICAL SYSTEM 134987
                                             POWER TRAIN 103631
                                        UNKNOWN OR OTHER 101773
                                                AIR BAGS  97746
                                                STEERING  76414
                                          SERVICE BRAKES  63571
                                  FUEL/PROPULSION SYSTEM  52658
                                          STRUCTURE:BODY  36957
                                   VEHICLE SPEED CONTROL  30574
                                              SUSPENSION  30329
                                       EXTERIOR LIGHTING  26833
                                        VISIBILITY/WIPER  21281
                      ELECTRONIC STABILITY CONTROL (ESC)  15691
FORWARD COLLIS

In [7]:
print(q("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT ODINO) AS distinct_complaints,
       ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT ODINO), 2) AS rows_per_complaint
FROM complaints
""").to_string(index=False))

   rows  distinct_complaints  rows_per_complaint
1103644               787344                 1.4


In [8]:
con.execute("""
CREATE OR REPLACE TABLE complaints AS
SELECT *, TRIM(split_part(component, ':', 1)) AS system FROM complaints
""")
con.execute("""
CREATE OR REPLACE TABLE recalls AS
SELECT *, TRIM(split_part(component, ':', 1)) AS system FROM recalls
""")

sys_c = q("SELECT system, COUNT(DISTINCT ODINO) AS complaints FROM complaints GROUP BY 1")
sys_r = q("SELECT system, COUNT(DISTINCT campaign) AS recalls FROM recalls GROUP BY 1")
both = sys_c.merge(sys_r, on="system", how="outer").fillna(0).sort_values("complaints", ascending=False)
print(both.to_string(index=False))

                                                     system  complaints  recalls
                                                     ENGINE    148471.0    132.0
                                          ELECTRICAL SYSTEM    140006.0   2031.0
                                                POWER TRAIN    106109.0    785.0
                                           UNKNOWN OR OTHER    101773.0      9.0
                                                   AIR BAGS     98249.0    853.0
                                                   STEERING     77191.0    720.0
                                             SERVICE BRAKES     63571.0    105.0
                                     FUEL/PROPULSION SYSTEM     52658.0      2.0
                                                  STRUCTURE     46875.0    998.0
                                      VEHICLE SPEED CONTROL     31646.0    109.0
                                                 SUSPENSION     31310.0    710.0
                            

/var/folders/jp/jh_70r2n0mv00gnhgr1w96tc0000gn/T/ipykernel_17167/2816448324.py:12: Pandas4Warning: 'int' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  both = sys_c.merge(sys_r, on="system", how="outer").fillna(0).sort_values("complaints", ascending=False)


In [9]:
print(q("""
WITH r AS (
    SELECT DISTINCT campaign, make, model, model_year, system, initiated_by FROM recalls
    WHERE recall_date >= DATE '2017-01-01'
),
c AS (
    SELECT DISTINCT make, model, model_year FROM complaints
)
SELECT
    r.initiated_by,
    COUNT(DISTINCT r.campaign) AS recalls,
    COUNT(DISTINCT CASE WHEN c.make IS NOT NULL THEN r.campaign END) AS vehicle_has_complaints,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN c.make IS NOT NULL THEN r.campaign END)
          / COUNT(DISTINCT r.campaign), 1) AS pct
FROM r LEFT JOIN c USING (make, model, model_year)
GROUP BY 1 ORDER BY 2 DESC
""").to_string(index=False))

initiated_by  recalls  vehicle_has_complaints  pct
         MFR     8112                    5437 67.0
         ODI      559                     466 83.4
        OVSC       61                      42 68.9


In [10]:
SYSTEM_MAP = {
    "ENGINE AND ENGINE COOLING": "ENGINE",
    "FUEL SYSTEM": "FUEL/PROPULSION SYSTEM",
    "HYBRID PROPULSION SYSTEM": "FUEL/PROPULSION SYSTEM",
    "ELECTRIC PROPULSION SYSTEM": "FUEL/PROPULSION SYSTEM",
    "VISIBILITY": "VISIBILITY/WIPER",
    "ELECTRONIC STABILITY CONTROL (ESC)": "ELECTRONIC STABILITY CONTROL",
}
DROP_SYSTEMS = ("UNKNOWN OR OTHER", "OTHER", "0", "")

map_sql = " ".join(f"WHEN '{k}' THEN '{v}'" for k, v in SYSTEM_MAP.items())

for tbl in ["complaints", "recalls"]:
    con.execute(f"""
    CREATE OR REPLACE TABLE {tbl} AS
    SELECT * EXCLUDE (sys_base),
           CASE sys_base {map_sql} ELSE sys_base END AS system_norm
    FROM (
        SELECT *, TRIM(split_part(system, ',', 1)) AS sys_base FROM {tbl}
    )
    """)

aligned = (q("SELECT system_norm, COUNT(DISTINCT ODINO) AS complaints FROM complaints GROUP BY 1")
           .merge(q("SELECT system_norm, COUNT(DISTINCT campaign) AS recalls FROM recalls GROUP BY 1"),
                  on="system_norm", how="outer").fillna(0)
           .sort_values("complaints", ascending=False))
print(aligned.head(25).to_string(index=False))

                 system_norm  complaints  recalls
                      ENGINE    160122.0    719.0
           ELECTRICAL SYSTEM    140006.0   2031.0
                 POWER TRAIN    106109.0    785.0
            UNKNOWN OR OTHER    101773.0      9.0
                    AIR BAGS     98249.0    853.0
                    STEERING     77191.0    720.0
              SERVICE BRAKES     71269.0   1051.0
      FUEL/PROPULSION SYSTEM     61762.0    822.0
                   STRUCTURE     46875.0    998.0
       VEHICLE SPEED CONTROL     31646.0    109.0
                  SUSPENSION     31310.0    710.0
           EXTERIOR LIGHTING     28565.0    698.0
            VISIBILITY/WIPER     27607.0    491.0
 FORWARD COLLISION AVOIDANCE     22801.0     77.0
ELECTRONIC STABILITY CONTROL     15713.0    114.0
                  SEAT BELTS     12912.0    481.0
                      WHEELS     12875.0    175.0
                       SEATS     11255.0    469.0
        BACK OVER PREVENTION      8042.0    263.0


/var/folders/jp/jh_70r2n0mv00gnhgr1w96tc0000gn/T/ipykernel_17167/2840330193.py:25: Pandas4Warning: 'int' is not supported as a fill value for str dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  on="system_norm", how="outer").fillna(0)


In [11]:
lopsided = aligned[((aligned["complaints"] > 1000) & (aligned["recalls"] < 10)) |
                   ((aligned["recalls"] > 50) & (aligned["complaints"] < 100))]
print("Still lopsided — check whether these need mapping:")
print(lopsided.to_string(index=False))

Still lopsided — check whether these need mapping:
     system_norm  complaints  recalls
UNKNOWN OR OTHER    101773.0      9.0


In [12]:
ceiling = q("""
WITH r AS (
    SELECT campaign, make, model, model_year, system_norm, initiated_by,
           MIN(recall_date) AS recall_date
    FROM recalls
    WHERE recall_date >= DATE '2017-01-01' AND system_norm NOT IN ('UNKNOWN OR OTHER','OTHER','0','')
    GROUP BY ALL
),
prior AS (
    SELECT r.campaign, r.initiated_by,
           COUNT(DISTINCT c.ODINO) AS prior_complaints
    FROM r
    LEFT JOIN complaints c
      ON c.make = r.make AND c.model = r.model AND c.model_year = r.model_year
     AND c.system_norm = r.system_norm
     AND c.received_date < r.recall_date
    GROUP BY 1, 2
),
per_campaign AS (
    SELECT campaign, initiated_by, MAX(prior_complaints) AS prior_complaints
    FROM prior GROUP BY 1, 2
)
SELECT initiated_by,
       COUNT(*) AS recalls,
       SUM(prior_complaints >= 1)  AS any_prior,
       SUM(prior_complaints >= 5)  AS five_plus,
       ROUND(100.0 * SUM(prior_complaints >= 5) / COUNT(*), 1) AS pct_five_plus
FROM per_campaign
GROUP BY 1 ORDER BY 2 DESC
""")
print(ceiling.to_string(index=False))

initiated_by  recalls  any_prior  five_plus  pct_five_plus
         MFR     8103     2165.0     1102.0           13.6
         ODI      559      417.0      358.0           64.0
        OVSC       61       27.0       16.0           26.2


In [13]:
con.close()